## About the dataset:

The EEG_data.csv contains the EEG data recorded from 10 students watching lecture videos, in motivation of detecting when they were confused.
Directly from the site, and I quote... "Content: These data are collected from ten students, each watching ten videos. Therefore, it can be seen as only 100 data points for these 12000+ rows. If you look at this way, then each data point consists of 120+ rows, which is sampled every 0.5 seconds (so each data point is a one minute video). Signals with higher frequency are reported as the mean value during each 0.5 second."

This is a challenging binary classification problem as it is described, as we want to make use of the following features to determine whether the student is confused or not confused.

* Examples of features include:

* SubjectID – identifies the student

* VideoID – identifies which lecture video was watched

* Attention – proprietary measure of focus

* Meditation – proprietary measure of calmness

* Raw EEG signal

* Delta band power (1–3 Hz)

* Theta band power (4–7 Hz)

* Alpha bands

* Beta bands

* Gamma bands


To start, let's read in the data...

In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv("EEG_data.csv")

# Inspect the data
df.head()
df.info()
df.describe()

df['user-definedlabeln'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12811 entries, 0 to 12810
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   SubjectID           12811 non-null  float64
 1   VideoID             12811 non-null  float64
 2   Attention           12811 non-null  float64
 3   Mediation           12811 non-null  float64
 4   Raw                 12811 non-null  float64
 5   Delta               12811 non-null  float64
 6   Theta               12811 non-null  float64
 7   Alpha1              12811 non-null  float64
 8   Alpha2              12811 non-null  float64
 9   Beta1               12811 non-null  float64
 10  Beta2               12811 non-null  float64
 11  Gamma1              12811 non-null  float64
 12  Gamma2              12811 non-null  float64
 13  predefinedlabel     12811 non-null  float64
 14  user-definedlabeln  12811 non-null  float64
dtypes: float64(15)
memory usage: 1.5 MB


user-definedlabeln
1.0    6567
0.0    6244
Name: count, dtype: int64

## Logistic Regression vs Neural Network

We will compare two models:

Logistic Regression (sklearn)

Neural Network (PyTorch)

## Prepare the Dataset

In [3]:
from sklearn.model_selection import train_test_split

# Drop non-feature columns
X = df.drop(['user-definedlabeln','predefinedlabel','SubjectID','VideoID'], axis=1)
y = df['user-definedlabeln']

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

## Logistic Regression

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

log_model = LogisticRegression(max_iter=2000)

log_model.fit(X_train, y_train)

y_pred = log_model.predict(X_test)

print("Logistic Regression Accuracy:",
      accuracy_score(y_test, y_pred))

Logistic Regression Accuracy: 0.5321888412017167


## Neural Network Model (using Pytorch)

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

# Convert to tensors
X_train_t = torch.tensor(X_train.values, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32)

X_test_t = torch.tensor(X_test.values, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32)

class Net(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16,1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

model = Net(X_train.shape[1])

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Training Loop:

In [6]:
for epoch in range(50):
    
    outputs = model(X_train_t).squeeze()
    
    loss = criterion(outputs, y_train_t)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

Evaluation:

In [7]:
with torch.no_grad():
    preds = model(X_test_t).squeeze()
    preds = (preds > 0.5).int()

accuracy = (preds == y_test_t.int()).float().mean()

print("Neural Network Accuracy:", accuracy.item())

Neural Network Accuracy: 0.5095590949058533


## Moving on to Feature Engineering...

We apply four techniques:

* Normalize features

* Remove outliers

* Select subset of features

* Select subset of subjects

First, normalize features (scaling the features for more stable training)

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

Second, we can remove outliers in the data via z-score filtering:

In [9]:
import numpy as np

z = np.abs((X - X.mean()) / X.std())

X_no_outliers = X[(z < 3).all(axis=1)]
y_no_outliers = y[X_no_outliers.index]

Third, we can do specific feature selection;
Some EEG bands may be more predictive:

In [11]:
important_features = [
    'Attention','Mediation',
    'Delta','Theta','Alpha1','Alpha2',
    'Beta1','Beta2','Gamma1','Gamma2'
]

X_selected = df[important_features]

Selection of a subset of subject IDs:

In [12]:
df_subset = df[df['SubjectID'] <= 5]

Evaluate on feature engineering vs. regular network:

In [14]:
# Helper function to train + evaluate neural network
def evaluate_nn(X, y, label):

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Convert to tensors
    X_train_t = torch.tensor(X_train.values if hasattr(X_train, "values") else X_train, dtype=torch.float32)
    X_test_t = torch.tensor(X_test.values if hasattr(X_test, "values") else X_test, dtype=torch.float32)

    y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
    y_test_t = torch.tensor(y_test.values, dtype=torch.float32)

    # Define neural network
    class Net(nn.Module):
        def __init__(self, input_size):
            super(Net, self).__init__()
            self.net = nn.Sequential(
                nn.Linear(input_size, 32),
                nn.ReLU(),
                nn.Linear(32, 16),
                nn.ReLU(),
                nn.Linear(16, 1),
                nn.Sigmoid()
            )

        def forward(self, x):
            return self.net(x)

    model = Net(X_train_t.shape[1])

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Train
    for epoch in range(50):
        outputs = model(X_train_t).squeeze()
        loss = criterion(outputs, y_train_t)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Evaluate
    with torch.no_grad():
        preds = model(X_test_t).squeeze()
        preds = (preds > 0.5).int()

    accuracy = (preds == y_test_t.int()).float().mean()

    print(f"{label} Accuracy:", accuracy.item())

In [16]:
# -------------------------
# Run Experiments
# -------------------------

# 1 Baseline (original data)
evaluate_nn(X, y, "Baseline")

# 2 Normalized features
evaluate_nn(X_scaled, y, "Normalized Features")

# 3 Outliers removed
evaluate_nn(X_no_outliers, y_no_outliers, "Outliers Removed")

# 4 Feature selection
evaluate_nn(X_selected, y, "Feature Selection")

# 5 Subject subset
X_subset = df_subset.drop(['user-definedlabeln','SubjectID','VideoID'], axis=1)
y_subset = df_subset['user-definedlabeln']

evaluate_nn(X_subset, y_subset, "SubjectID Subset")

Baseline Accuracy: 0.5072181224822998
Normalized Features Accuracy: 0.6071010828018188
Outliers Removed Accuracy: 0.495071679353714
Feature Selection Accuracy: 0.4935622215270996
SubjectID Subset Accuracy: 0.5207253694534302
